In [10]:
sys.path.append("../dataloaders")

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from data_loader_cropping_padding import CroppedPaddedDataset
from torch.utils.data import random_split, DataLoader
import matplotlib.pyplot as plt
import sys

In [12]:
dataset = CroppedPaddedDataset("../labels.csv", "../images")

Image width 400 and height 200
Found 402 valid samples


In [13]:
class CNNRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, 1)  
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = self.regressor(x)
        return x

In [14]:
#Training and validation data
val_split = 0.2  
n_total = len(dataset)
n_val = int(n_total * val_split)
n_train = n_total - n_val

train_set, val_set = random_split(dataset, [n_train, n_val])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False)

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = CNNRegressor().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)


Using device: cpu


In [ ]:
num_epochs = 50
train_mse_list = []
val_mse_list = []
val_mae_list = []

for epoch in range(num_epochs):
    #Training
    model.train()
    running_loss = 0.0
    n = 0

    for imgs, labels, names in train_loader:   
        imgs = imgs.to(device)
        labels = labels.float().view(-1, 1).to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size = imgs.size(0)
        running_loss += loss.item() * imgs.size(0)
        n += batch_size

    print(f"Epoch {epoch+1}/{num_epochs} - Train MSE: {running_loss / n:.4f}")

    #Validation
    model.eval()
    val_loss = 0.0
    val_mae = 0.0
    n_val = 0

    with torch.no_grad():
        for imgs, labels, names in val_loader:
            imgs = imgs.to(device)
            labels = labels.float().view(-1, 1).to(device)

            outputs = model(imgs)

            #MSE 
            loss = criterion(outputs, labels)
            val_loss += loss.item() * imgs.size(0)

            #MAE 
            val_mae += torch.abs(outputs - labels).sum().item()

            n_val += imgs.size(0)

    train_mse = running_loss / n
    val_mse = val_loss / n_val
    val_mae_epoch = val_mae / n_val

    train_mse_list.append(train_mse)
    val_mse_list.append(val_mse)
    val_mae_list.append(val_mae_epoch)

    print(f"Val MSE: {val_loss / n_val:.4f} | Val MAE: {val_mae / n_val:.4f}")

Epoch 1/50 - Train MSE: 66.4229
Val MSE: 74.8487 | Val MAE: 5.9649
Epoch 2/50 - Train MSE: 64.9547
Val MSE: 72.6938 | Val MAE: 5.7964
Epoch 3/50 - Train MSE: 62.2219
Val MSE: 68.7647 | Val MAE: 5.4876
Epoch 4/50 - Train MSE: 57.6379
Val MSE: 61.4618 | Val MAE: 4.8795


In [ ]:
epochs = range(1, num_epochs + 1)

plt.plot(epochs, train_mse_list, label='Train MSE')
plt.plot(epochs, val_mse_list, label='Val MSE')
plt.xlabel('Epoch')
plt.ylabel('Error')
plt.legend()
plt.title('Training and Validation MSE')
plt.show()

In [ ]:
epochs = range(1, num_epochs + 1)

plt.plot(epochs, val_mae_list, label='Val MAE')
plt.xlabel('Epoch')
plt.ylabel('Error')
plt.legend()
plt.title('Validation MAE')
plt.show()